# 이진 분류 실험 ⑪-1 — 다운샘플링 데이터셋 다시 만들기 (select)
서버를 옮기면서 사라진 `parquets/HI-Large/sampling/select/` 를 다시 만든다. 학습은 여기서 하지 않는다.

- 만드는 것: `select/<이름>/selected.parquet` = **학습 구간 양성 전부 + 선택된 정상 거래** 의 `edge_id` 목록 (+ `meta.json`)
- 만드는 코드: 예전 샘플링 실험의 `code/05_sampling_select.py` 를 그대로 부른다 (코드 수정 없음, 시드 42 고정 → `transactions.parquet` 가 같으면 예전과 같은 목록)
- 분할: 학습 2022-08-08 ~ 10-02 (`sampling_lib.TRAIN` = 이진분류 실험과 같은 분할). 검증 · 테스트 구간은 건드리지 않는다
- 적용 위치: **loss** — 그래프 · 이웃은 전체 그대로 두고, 다음 노트북에서 이 목록을 학습 target 으로만 쓴다

| 조건 | 내용 | 필요한 파일 | 예전 결과 (검증 AP, v10 구조) |
|---|---|---|---|
| `random_r10` | 날마다 정상 : 양성 = 10 : 1 무작위 (RUS) | `transactions.parquet`, `patterns.parquet` | 0.496 ± 0.059 (시드 3) |
| `hard_r10` | 예산의 50% 를 세탁 계좌 주변 정상으로 | 〃 | 0.493 ± 0.036 (시드 3) |
| `strata_r10_day-amount-cross_bank` | 날짜 · 금액 · 은행 간 3축 층화 | + `features_train/edge_features.parquet` (v10 피처) | 0.501 (시드 1) |

기본은 `random_r10` 하나만 만든다. 나머지는 1 절에서 주석을 풀면 된다. 이미 있는 조건은 건너뛴다

## 0. 작업 준비

In [ ]:
import sys, json, importlib
from pathlib import Path
import numpy as np
import pandas as pd
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 40)

sys.path.append(str(Path.cwd()))
import bc_lib as B                                      # code/ 를 sys.path 에 넣어 준다 (sampling_lib, 05_sampling_select)
L = B.L
select = importlib.import_module('05_sampling_select')

In [ ]:
# ---- 설정 ------------------------------------------------------------------
THREADS, MEMORY = 8, '12GB'           # DuckDB 한도 (지금까지 노트북과 같은 값)
R = '10'                              # 정상 : 양성 비율
TRAIN = L.TRAIN                       # 학습 구간
assert TRAIN == B.WINDOWS['HI-Large']['train'] == '2022-08-08:2022-10-02', '이진분류 실험의 분할과 다르다'

SELECT_DIR = L.SAMPLING / 'select'
print('ROOT', L.ROOT); print('출력', SELECT_DIR); print('학습 구간', TRAIN)
for f in ['transactions.parquet', 'patterns.parquet']:
    print(f"{'OK  ' if (L.BASE / f).is_file() else '없음'} {f}")
print(f"{'OK  ' if (L.BASE / L.FEAT['train'] / 'edge_features.parquet').is_file() else '없음'} {L.FEAT['train']}/edge_features.parquet  (strata 만 필요)")

## 1. 선택 (select)
조건 하나에 몇 분 걸린다 (학습 구간 1.02억 건의 라벨을 읽어 날짜별로 뽑는다). 끝나면 `양성 … + 정상 …` 한 줄이 찍힌다

In [ ]:
common = ['--train', TRAIN, '--threads', str(THREADS), '--memory', MEMORY]

select.main(['--method', 'random', '--ratio', R] + common)                                   # RUS — 보고서 기본값

# 더 볼 조건 — 필요하면 주석을 푼다
# select.main(['--method', 'hard', '--ratio', R, '--hard-frac', '0.5'] + common)             # 세탁 계좌 주변 정상 50%
# select.main(['--method', 'strata', '--ratio', R, '--axes', 'day,amount,cross_bank', '--features', L.FEAT['train']] + common)   # features_train 필요

## 2. 확인
만들어진 조건을 전부 읽어 수를 본다. 예전 보고서(`sampling/report.md`) 의 학습 구간 수치와 같아야 한다

| 항목 | 기대값 |
|---|---:|
| 양성 | 110,671 |
| 학습 구간 정상 (모집단) | 102,225,958 |
| `random_r10` 의 정상 | 1,106,710 (= 양성 × 10) |

In [ ]:
EXPECT_POS, EXPECT_NEG_POP = 110_671, 102_225_958        # sampling/report.md '학습 구간 데이터'
metas = {p.parent.name: json.loads(p.read_text(encoding='utf-8')) for p in sorted(SELECT_DIR.glob('*/meta.json'))}
assert metas, 'select 결과가 없다 — 1 절을 먼저 돌린다'
tbl = pd.DataFrame(metas).T[['method', 'ratio', 'budget', 'seed', 'rows', 'pos', 'neg', 'neg_population', 'daily_keep_min', 'daily_keep_max', 'seconds']]
tbl['정상:양성'] = (tbl['neg'] / tbl['pos']).astype(float).round(2)
tbl['양성 수 일치'] = tbl['pos'] == EXPECT_POS
tbl['모집단 일치'] = tbl['neg_population'] == EXPECT_NEG_POP
tbl

In [ ]:
# 조건 하나를 열어 본다 — 열 구성, 양성이 전부 들었는지, 중복이 없는지
NAME = f'random_r{R}'
sel = pd.read_parquet(SELECT_DIR / NAME / 'selected.parquet')
assert sel['edge_id'].is_unique and sel['edge_id'].is_monotonic_increasing, 'edge_id 가 중복이거나 정렬이 안 됐다'
assert int(sel['y'].sum()) == metas[NAME]['pos'], '양성 수가 meta 와 다르다'
print(f"{NAME}: {len(sel):,}행 · 양성 {int(sel['y'].sum()):,} · 정상 {int((sel['y'] == 0).sum()):,} · 기간 {sel['day'].min()} ~ {sel['day'].max()}")
sel.head()

In [ ]:
# 날짜별 양성 / 정상 수 — random(pos 규칙) 은 날마다 정상 = 양성 × R 이어야 한다
daily = sel.groupby('day')['y'].agg(pos='sum', n='size'); daily['neg'] = daily['n'] - daily['pos']
daily['neg_per_pos'] = (daily['neg'] / daily['pos']).round(2)
print(f"날짜 {len(daily)}일 · 날짜별 정상:양성 최소 {daily['neg_per_pos'].min()} ~ 최대 {daily['neg_per_pos'].max()}")
daily[['pos', 'neg', 'neg_per_pos']].head(10)

## 3. 다음 노트북에서 쓰는 법
그래프는 그대로 두고 학습 target 만 이 목록으로 바꾼다 (loss 샘플링). 09 의 `train_rows = np.flatnonzero(in_train)` 자리에 아래를 넣는다

```python
sel = pd.read_parquet(L.SAMPLING / 'select' / 'random_r10' / 'selected.parquet', columns=['edge_id'])
train_rows = B.rows_of(tx['edge_id'].to_numpy(), sel['edge_id'].to_numpy())     # 샘플링 없음: np.flatnonzero(in_train)
```